## Google Street View data retrival

Load configuration, define reusable functions, then run the target range.

### Setup

In [1]:
import importlib
import os
import sys
from pathlib import Path


def load_env_file(filename=".env"):
    """Load KEY=VALUE pairs from the nearest .env file into os.environ."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        env_path = base / filename
        if not env_path.is_file():
            continue

        for raw_line in env_path.read_text().splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue

            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip("\"'")
            os.environ.setdefault(key, value)

        return env_path

    return None


def find_project_root():
    """Find the repository root by locating data/cleaned_data.csv."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "cleaned_data.csv").is_file():
            return base

    raise FileNotFoundError("Could not locate data/cleaned_data.csv from the current notebook directory.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import gsv_pipeline
gsv_pipeline = importlib.reload(gsv_pipeline)

DATA_PATH = PROJECT_ROOT / "data" / "cleaned_data.csv"

RUN_CONFIG = {
    "start": 101,
    "end": 200,
    "out_dir": PROJECT_ROOT / "notebook" / "gsv_out" / "cleaned_data_first50",
    "target_date_field": "create_date",
    "candidate_radius": 25,
    "max_distance_meters": 20,
    "max_baseline_shift_meters": 10,
    "require_baseline": True,
    "allow_distance_fallback": False,
}

ENV_PATH = load_env_file()
API_KEY = os.environ.get("GSV_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "Missing GSV_API_KEY. Copy .env.example to .env, use a key with both Street View Static API and Geocoding API enabled, and rerun this cell."
    )

{
    "env_path": str(ENV_PATH) if ENV_PATH else None,
    "data_path": str(DATA_PATH),
    "run_config": {k: str(v) if isinstance(v, Path) else v for k, v in RUN_CONFIG.items()},
}

{'env_path': '/Volumes/D/Leo/Study/CS Capstone/UrbanVision/.env',
 'data_path': '/Volumes/D/Leo/Study/CS Capstone/UrbanVision/data/cleaned_data.csv',
 'run_config': {'start': 101,
  'end': 200,
  'out_dir': '/Volumes/D/Leo/Study/CS Capstone/UrbanVision/notebook/gsv_out/cleaned_data_first50',
  'target_date_field': 'create_date',
  'candidate_radius': 25,
  'max_distance_meters': 20,
  'max_baseline_shift_meters': 10,
  'require_baseline': True,
  'allow_distance_fallback': False}}

### Run a range

In [2]:
results = gsv_pipeline.fetch_entries(
    csv_path=DATA_PATH,
    api_key=API_KEY,
    start=RUN_CONFIG["start"],
    end=RUN_CONFIG["end"],
    out_dir=RUN_CONFIG["out_dir"],
    target_date_field=RUN_CONFIG["target_date_field"],
    candidate_radius=RUN_CONFIG["candidate_radius"],
    max_distance_meters=RUN_CONFIG["max_distance_meters"],
    max_baseline_shift_meters=RUN_CONFIG["max_baseline_shift_meters"],
    require_baseline=RUN_CONFIG["require_baseline"],
    allow_distance_fallback=RUN_CONFIG["allow_distance_fallback"],
)

downloaded = [item for item in results if item.get("ok")]
failed = [item for item in results if not item.get("ok")]

{
    "requested_range": [RUN_CONFIG["start"], RUN_CONFIG["end"]],
    "requested_entries": RUN_CONFIG["end"] - RUN_CONFIG["start"] + 1,
    "downloaded_images": len(downloaded),
    "failed_entries": len(failed),
    "sample_results": results[:5],
}

{'requested_range': [101, 200],
 'requested_entries': 100,
 'downloaded_images': 67,
 'failed_entries': 33,
 'sample_results': [{'index': 101,
   '_id': '21183',
   'parcel_id': '0010D00258000000',
   'address': 'No primary address specified',
   'target_capture_date': '2021-04-22',
   'ok': True,
   'status': 'OK',
   'review_summary': {'selected_pano_id': '0cTVvKgZkMNruBXCJl98Aw',
    'selected_pano_date': '2016-07',
    'target_capture_date': '2021-04-22',
    'temporal_gap_months': 57,
    'temporal_gap_years': 4.75,
    'selected_distance_to_target_meters': 16.150159447188116,
    'selected_distance_to_baseline_meters': 0.0,
    'baseline_pano_id': '0cTVvKgZkMNruBXCJl98Aw',
    'baseline_pano_date': '2016-07',
    'baseline_distance_from_target_meters': 16.150159447188116,
    'candidate_radius_meters': 25,
    'distance_guardrail_meters': 20,
    'baseline_shift_guardrail_meters': 10,
    'within_target_distance_guardrail': True,
    'within_baseline_shift_guardrail': True,
    '